# Embedding / Storing / Retrieving

In [ ]:
from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# from langchain_huggingface import HuggingFaceEmbeddings
# from langchain_ollama.llms import OllamaLLM # run ollama models
# from langchain_ollama import OllamaEmbeddings  # Ollama embeddings for vectorization

from langchain.retrievers import ParentDocumentRetriever
from langchain.text_splitter import CharacterTextSplitter # split document based on characters.
# from langchain_text_splitters import CharacterTextSplitter # split document based on characters.

from langchain.storage import InMemoryStore

from IPython.display import display, HTML, Markdown


In [2]:
load_dotenv(override=True)

True

In [16]:
# Model configuration
openai_llm = ChatOpenAI(
    model="gpt-4.1-nano",
)

## Embedding Models

In [3]:
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# embeddings = OllamaEmbeddings( model="mxbai-embed-large", base_url="http://localhost:11434") # base_url:ju Set the Ollama server URL

In [4]:
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")
document = loader.load() # Download the PDF and Load as a langchain document

text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")
chunks = text_splitter.split_documents(document)

print(len(chunks)) # Length of chunks

148


In [ ]:
texts = [chunk.page_content for chunk in chunks]

embedding_result = embeddings.embed_documents(texts) # embeds content in each of the chunks into vectors

print("Dimension of each chunk's embedding:", len(embedding_result[0]))
print(embedding_result[0][:5]) # first 5 numbers in the vector representation of first chunk's content

Dimension of each chunk's embedding: 1536
[0.012022854760289192, -0.023233922198414803, 0.03795807063579559, 0.04971499368548393, -0.04448036476969719]


## Vector stores - Databases

In [5]:
from langchain.vectorstores import Chroma
# from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings) # create vectorstore in memory
# vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory="langchain-papers-db")
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

query = "Langchain"
docs = vectorstore.similarity_search(query)
print(docs[0].page_content)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vectorstore created with 148 documents


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[4] LangChain, https://www.langchain.com/  (accessed Nov. 29, 202 3). 
[5] LangChain ChatModels, https://blog.langchain.dev/chat -models/ 
(accessed Nov. 29, 2023).


## Retrievers

### Vector store-backed retrievers

Vector store retrievers are retrievers that use a vector store to retrieve documents. 

In [20]:
retriever = vectorstore.as_retriever() # This converts the vector store into a retriever interface that can fetch relevant documents

docs = retriever.invoke("Langchain") # Invoke the retriever with the query "Langchain"

docs[0] # Access the first (most relevant) document from the retrieval results

Document(metadata={'page': 5, 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf'}, page_content='[4] LangChain, https://www.langchain.com/  (accessed Nov. 29, 202 3). \n[5] LangChain ChatModels, https://blog.langchain.dev/chat -models/ \n(accessed Nov. 29, 2023).')

`Note that the results are identical to the results you obtained using the similarity search strategy.`

### Parent document retrievers

During retrieval, this retriever first fetches the small chunks, but then looks up the parent IDs for the data and returns those larger documents.

In [ ]:
# Set up two different text splitters for a hierarchical splitting approach:

# 1. Parent splitter creates larger chunks (2000 characters) - This is used to split documents into larger, more contextually complete sections
parent_splitter = CharacterTextSplitter(chunk_size=2000, chunk_overlap=20, separator='\n')

# 2. Child splitter creates smaller chunks (400 characters) - This is used to split the parent chunks into smaller pieces for more precise retrieval
child_splitter = CharacterTextSplitter(chunk_size=400, chunk_overlap=20, separator='\n')

# Create a Chroma vector store with a collection name and embedding function
vectorstore = Chroma(
    collection_name="split_parents", embedding_function=embeddings
)

# Set up an in-memory storage layer for the parent documents
# This will store the larger chunks that provide context, but won't be directly embedded
store = InMemoryStore()

/tmp/ipykernel_200878/2234862490.py:10: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 0.4. An updated version of the class exists in the langchain-chroma package and should be used instead. To use it run `pip install -U langchain-chroma` and import as `from langchain_chroma import Chroma`.
  vectorstore = Chroma(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [9]:
# Create a ParentDocumentRetriever instance that implements hierarchical document retrieval
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore, # child document embeddings will be stored and searched    
    docstore=store, # parent documents will be stored - These larger chunks won't be embedded but will be retrieved by ID when needed
    child_splitter=child_splitter, # The splitter used to create small chunks (400 chars) for precise vector search
    parent_splitter=parent_splitter, # The splitter used to create larger chunks (2000 chars) for better context
)

In [ ]:
retriever.add_documents(document) # add documents to the hierarchical retrieval system

In [11]:
len(list(store.yield_keys())) # retrieves and counts the number of parent document IDs stored in the document store

16

In [13]:
sub_docs = vectorstore.similarity_search("Langchain") # Verify, underlying vector store still retrieves the small chunks.
print(sub_docs[0].page_content)

LangChain helps us to unlock the ability to harness the 
LLM’s immense potential in tasks such as document analysis, 
chatbot development, code analysis, and countless other 
applications. Whether your desire is to unlock deeper natural 
language understanding , enhance data, or circumvent 
language barriers through translation, LangChain is ready to


In [14]:
retrieved_docs = retriever.invoke("Langchain") # retrieve the relevant large chunk.
print(retrieved_docs[0].page_content)

LangChain helps us to unlock the ability to harness the 
LLM’s immense potential in tasks such as document analysis, 
chatbot development, code analysis, and countless other 
applications. Whether your desire is to unlock deeper natural 
language understanding , enhance data, or circumvent 
language barriers through translation, LangChain is ready to 
provide the tools and programming support you need to do 
without it that it is not only difficult but also fresh for you . Its 
core functionalities encompass:  
1. Context -Aware Capabilities: LangChain facilitates the 
development of applications that are inherently 
context -aware. This means that these applications can 
connect to a language model and draw from various 
sources of context, such as prompt instructions, a  few-
shot examples, or existing content, to ground their 
responses effectively.  
2. Reasoning Abilities: LangChain equips applications 
with the capacity to reason effectively. By relying on a 
language model, thes

## RetrievalQA - RAG

create a QA bot that can answer your questions based on the paper.

`A Basic example of RAG`

In [17]:
from langchain.chains import RetrievalQA

# Create a RetrievalQA chain
qa = RetrievalQA.from_chain_type(
    llm=openai_llm, # llm for generating answers
    chain_type="stuff", # "stuff" means all retrieved documents are simply concatenated and passed to the LLM
    retriever=vectorstore.as_retriever(), # converts the vector store into a retriever interface; to fetch relevant documents
    return_source_documents=False # Whether to include the source documents in the response - false means: return only the generated answer
)

query = "what is this paper discussing?"

# Execute the QA chain with the query
# This will:
# 1. Send the query to the retriever to get relevant documents
# 2. Combine those documents using the "stuff" method
# 3. Send the query and combined documents to the LLM
# 4. Return the generated answer (without source documents)
qa.invoke(query)

{'query': 'what is this paper discussing?',
 'result': "This paper discusses the development of an advanced mental health support system that leverages various AI and machine learning tools, such as Lang Chain's ChatPrompt Template, HumanMessage Prompt Template, ConversationBufferMemory, and LLMChain, for early detection and comprehensive support of mental health issues. It emphasizes the importance of early identification of mental health challenges like anxiety, depression, and suicidal thoughts, and describes how recent deep learning advancements can aid in this effort. Additionally, the paper mentions the implementation of Streamlit to improve user interaction with the chatbot."}

## Exercise - Building a Simple Retrieval System with LangChain

In [ ]:
# 1. Load a document about AI
loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/") # Scrap a page from Website and convert it into langchain document
documents = loader.load()

# 2. Split the document into chunks
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")
chunks = text_splitter.split_documents(documents)
print("Chunks Length: ", len(chunks)) # Length of chunks

# 3. Initiate embedding model
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# 4. Create a vector store
vector_store = Chroma.from_documents(documents=chunks, embedding=embedding_model)

# 5. Create a retriever
retriever = vector_store.as_retriever()

# 6. Search Document based on query
def search_documents(query, top_k=3):
    """Search for documents relevant to a query"""
    docs = retriever.get_relevant_documents(query) # Use the retriever to get relevant documents
    return docs[:top_k] # Limit to top_k if specified

# 7. Test with a few queries
test_queries = [
    "What is LangChain?",
    "How do retrievers work?",
    "Why is document splitting important?"
]

for query in test_queries:
    print(f"\nQuery: {query}")
    results = search_documents(query)
    print(f"---------------Result of Query: {query}----------------------")
    print(results)
    print()
    



Created a chunk of size 1285, which is longer than the specified 200
Created a chunk of size 323, which is longer than the specified 200
Created a chunk of size 205, which is longer than the specified 200
Created a chunk of size 928, which is longer than the specified 200
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Chunks Length:  10

Query: What is LangChain?


/tmp/ipykernel_200878/382118377.py:24: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use invoke instead.
  docs = retriever.get_relevant_documents(query) # Use the retriever to get relevant documents


---------------Result of Query: What is LangChain?----------------------
[Document(metadata={'page': 5, 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf'}, page_content='[4] LangChain, https://www.langchain.com/  (accessed Nov. 29, 202 3). \n[5] LangChain ChatModels, https://blog.langchain.dev/chat -models/ \n(accessed Nov. 29, 2023).'), Document(metadata={'description': 'LangChain is an open source framework with a pre-built agent architecture and integrations for any model or tool — so you can build agents that adapt as fast as the ecosystem evolves', 'language': 'en', 'source': 'https://python.langchain.com/v0.2/docs/introduction/', 'title': 'LangChain overview - Docs by LangChain'}, page_content='LangChain agents are built on top of LangGraph in order to provide durable execution, streaming, human-in-the-loop, persistence, and more. You do not need to know LangGraph for basic LangChain agent usage.'), Document(